In [1]:
# Pytorch packages
import timm
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

# Other packages
import os
import json
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder

In [4]:
# Hyperparameters
MODEL_NAME = "convnext_nano.in12k"
DEVICE_ID = 0
BATCH_SIZE = 512
IMAGE_SIZE = 224
DATA_DIR = r"/home/c/choton/beemachine/datasets/Beemachine_Partwhole_v5/"
MODEL_WEIGHTS = r'/home/c/choton/beemachine/codes/AG_vision_2026/1_classification/Beemachine/convnext_nano.in12k_timm_new_dataset_logs/checkpoints/best_model.pth'

# Load the train and validation dataset
train_datapath = os.path.join(DATA_DIR, 'train_aug_labels.csv')
val_datapath = os.path.join(DATA_DIR, 'val_labels.csv')
test_datapath = os.path.join(DATA_DIR, 'test_labels.csv')

train_df = pd.read_csv(train_datapath) 
val_df = pd.read_csv(val_datapath) 
test_df = pd.read_csv(test_datapath)

le = LabelEncoder()
train_df["label"] = le.fit_transform(train_df["species"])
val_df["label"] = le.transform(val_df["species"])
test_df["label"] = le.transform(test_df["species"])
num_classes = len(le.classes_)
print(f"Total images, Train: {len(train_df['label'])}, Validation: {len(val_df['label'])}, Test: {len(test_df['label'])}")
print(f"Total classes: {num_classes}")

train_path = os.path.join(DATA_DIR, "train", 'aug_images') # Path for the training data
val_path = os.path.join(DATA_DIR, "valid", 'images') # Path for validation data
test_path = os.path.join(DATA_DIR, "test", 'images') # Path for testing data

Total images, Train: 34722, Validation: 1158, Test: 771
Total classes: 160


In [15]:
class SpeciesDataset(Dataset):
    def __init__(self, df, img_dir, image_size):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.4925, 0.4475, 0.3490), # Custom normalization values for beemachine_small_2025_v3 (segmentation) dataset
                             std=(0.2392, 0.2265, 0.2213))
        # transforms.Normalize(mean=[0.485, 0.456, 0.406], # Imagenet Normalization values
        #              std=[0.229, 0.224, 0.225])
    ])
        self.num_classes = self.df["label"].nunique()
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.loc[idx, "image"]
        label = self.df.loc[idx, "label"]

        img_path = os.path.join(self.img_dir, img_name)
        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label

In [16]:
train_dataset = SpeciesDataset(df=train_df, img_dir=train_path, image_size=IMAGE_SIZE)
val_dataset = SpeciesDataset(df=val_df, img_dir=val_path, image_size=IMAGE_SIZE)
test_dataset = SpeciesDataset(df=test_df, img_dir=test_path, image_size=IMAGE_SIZE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

num_classes = train_dataset.num_classes
print(f"Classes: {num_classes} | Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

Classes: 160 | Train: 34722 | Val: 1158 | Test: 771


In [17]:
device = torch.device(f"cuda:{DEVICE_ID}")
state_dict = torch.load(MODEL_WEIGHTS, map_location=device)
criterion = nn.CrossEntropyLoss()  #label_smoothing=0.1)
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=num_classes)
model.to(device)
model.load_state_dict(state_dict)
model.eval()

ConvNeXt(
  (stem): Sequential(
    (0): Conv2d(3, 80, kernel_size=(4, 4), stride=(4, 4))
    (1): LayerNorm2d((80,), eps=1e-06, elementwise_affine=True)
  )
  (stages): Sequential(
    (0): ConvNeXtStage(
      (downsample): Identity()
      (blocks): Sequential(
        (0): ConvNeXtBlock(
          (conv_dw): Conv2d(80, 80, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=80)
          (norm): LayerNorm2d((80,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Conv2d(80, 320, kernel_size=(1, 1), stride=(1, 1))
            (act): GELU()
            (drop1): Dropout(p=0.0, inplace=False)
            (norm): Identity()
            (fc2): Conv2d(320, 80, kernel_size=(1, 1), stride=(1, 1))
            (drop2): Dropout(p=0.0, inplace=False)
          )
          (shortcut): Identity()
          (drop_path): Identity()
        )
        (1): ConvNeXtBlock(
          (conv_dw): Conv2d(80, 80, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=80)


In [18]:
def validate_one_epoch(model, val_loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        pbar = tqdm(val_loader, desc="[Val]", position=0, leave=False)
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

            pbar.set_postfix(loss=loss.item())
    pbar.close()
    avg_loss = running_loss / total
    accuracy = correct / total
    return avg_loss, accuracy

In [19]:
test_loss, test_acc = validate_one_epoch(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

Test Loss: 1.2906 | Test Acc: 0.6744


In [21]:
img, label = test_dataset[100]
outputs = model(img.unsqueeze(0).to(device))
_, preds = outputs.max(1)
preds, label, preds.eq(label)

(tensor([49], device='cuda:0'), 49, tensor([True], device='cuda:0'))

In [22]:
decoded_labels = le.inverse_transform(test_df["label"])
decoded_labels[label]

'Bombus_flavidus'